# Combined-MA ONNX evaluation

이 노트북은 **학습을 다시 하지 않습니다.**

목적:
1. 여러 MA sample을 합쳐서 학습한 **하나의 combined-MA ONNX model**을 읽습니다.
2. 동일한 ONNX를 MA = 18, 25, 36, 50, 70, 90, 110, 125 GeV signal에 각각 적용합니다.
3. TTLJ에도 동일한 ONNX를 적용합니다.
4. mass point별 ROC/AUC, 전체 MA ROC, AUC-vs-MA를 확인합니다.
5. PNG와 PDF를 동시에 저장합니다.

중요: ONNX inference에서는 training 때 저장한 `*_features.json`의 feature 순서를 그대로 사용합니다.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import ROOT
import matplotlib.pyplot as plt
import onnxruntime as ort

from sklearn.metrics import roc_curve, roc_auc_score

np.random.seed(42)


In [ ]:
# ============================================================
# 2. Configuration
#    우선 이 셀의 경로/모델 이름만 확인하세요.
# ============================================================
TREE_NAME = "Training_Tree"

BASE_DIR = "/data9/Users/bhoh/SKNanoOutput/AtobbMLTree/2024"
MODEL_DIR = "/data9/Users/eunsu/MachineLearning/models"

TARGET_MHC = 130
MA_VALUES = [18, 25, 36, 50, 70, 90, 110, 125]

# 실제 combined-MA training notebook에서 저장한 이름으로 수정
MODEL_NAME = "xgboost_atobb_MHc130_MAcombined_ttlJ"

ONNX_PATH = f"{MODEL_DIR}/{MODEL_NAME}.onnx"
FEATURE_JSON_PATH = f"{MODEL_DIR}/{MODEL_NAME}_features.json"
SCALER_PICKLE_PATH = f"{MODEL_DIR}/{MODEL_NAME}_scaler.pkl"
SCALER_JSON_PATH = f"{MODEL_DIR}/{MODEL_NAME}_scaler.json"

SIG_FILES = {
    ma: f"{BASE_DIR}/TTToHcToWAToBB-MHc{TARGET_MHC}_MA{ma}_SingleLepFilter.root"
    for ma in MA_VALUES
}
BKG_FILE = f"{BASE_DIR}/TTLJ_powheg.root"

# None = evaluation에서 TTLJ 전체 사용
N_BKG_EVAL = None
WEIGHT_COL = "weight_train"

PLOT_DIR = Path(f"./onnx_eval_{MODEL_NAME}")
PDF_DIR = PLOT_DIR / "pdf"
OUTPUT_DIR = PLOT_DIR / "outputs"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ONNX :", ONNX_PATH, os.path.exists(ONNX_PATH))
print("JSON :", FEATURE_JSON_PATH, os.path.exists(FEATURE_JSON_PATH))
print("BKG  :", BKG_FILE, os.path.exists(BKG_FILE))
print("\nSignal files:")
for ma, path in SIG_FILES.items():
    print(f"  MA={ma:3d}: exists={os.path.exists(path)}  {path}")


In [ ]:
# ============================================================
# 3. Load exact feature order saved during training
# ============================================================
if not os.path.exists(ONNX_PATH):
    raise FileNotFoundError(
        f"ONNX model not found: {ONNX_PATH}\n"
        "MODEL_NAME / MODEL_DIR를 실제 combined-MA model에 맞게 수정하세요."
    )

if not os.path.exists(FEATURE_JSON_PATH):
    raise FileNotFoundError(
        f"Feature metadata not found: {FEATURE_JSON_PATH}"
    )

with open(FEATURE_JSON_PATH) as f:
    metadata = json.load(f)

FEATURES = metadata["features"]
SCALER_TYPE = metadata.get("scaler_type", None)

print("Number of features:", len(FEATURES))
print("Scaler type:", SCALER_TYPE)
print()
for i, feat in enumerate(FEATURES):
    print(f"{i:02d}: {feat}")


In [ ]:
# ============================================================
# 4. Load scaler only if training used one
# ============================================================
scaler = None
scaler_meta = None

if SCALER_TYPE is not None:
    if os.path.exists(SCALER_PICKLE_PATH):
        with open(SCALER_PICKLE_PATH, "rb") as f:
            scaler = pickle.load(f)
        print("Loaded scaler:", SCALER_PICKLE_PATH)

    elif os.path.exists(SCALER_JSON_PATH):
        with open(SCALER_JSON_PATH) as f:
            scaler_meta = json.load(f)
        print("Using scaler JSON:", SCALER_JSON_PATH)

    else:
        raise FileNotFoundError(
            "Model metadata says a scaler was used, but scaler file was not found."
        )
else:
    print("No scaler: raw features will be passed to ONNX.")


In [ ]:
# ============================================================
# 5. ROOT -> pandas helper
# ============================================================
def check_required_branches(root_path, tree_name, branches):
    f = ROOT.TFile.Open(root_path, "READ")
    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open ROOT file: {root_path}")

    tree = f.Get(tree_name)
    if not tree:
        f.Close()
        raise RuntimeError(f"Cannot find tree '{tree_name}' in {root_path}")

    available = {b.GetName() for b in tree.GetListOfBranches()}
    missing = [x for x in branches if x not in available]
    f.Close()

    if missing:
        raise RuntimeError(
            f"Missing branches in {root_path}:\n" + "\n".join(missing)
        )

def read_root(root_path, tree_name, branches):
    check_required_branches(root_path, tree_name, branches)
    rdf = ROOT.RDataFrame(tree_name, root_path)
    arrays = rdf.AsNumpy(branches)
    return pd.DataFrame({k: np.asarray(v) for k, v in arrays.items()})


In [ ]:
# ============================================================
# 6. Read ALL evaluation samples
#
# 여기서는 train/validation split을 다시 하지 않습니다.
# 이미 만들어진 ONNX를 각 sample에 적용합니다.
# ============================================================
BRANCHES_TO_READ = list(dict.fromkeys(FEATURES + [WEIGHT_COL]))

sig_dfs = {}

for ma, path in SIG_FILES.items():
    print(f"Reading signal MA={ma} ...")
    df = read_root(path, TREE_NAME, BRANCHES_TO_READ)
    df["mass_point"] = ma
    df["label"] = 1
    sig_dfs[ma] = df
    print("  entries =", len(df))

print("\nReading TTLJ ...")
df_bkg = read_root(BKG_FILE, TREE_NAME, BRANCHES_TO_READ)
df_bkg["label"] = 0

if N_BKG_EVAL is not None and len(df_bkg) > N_BKG_EVAL:
    df_bkg = df_bkg.sample(
        n=N_BKG_EVAL, random_state=42
    ).reset_index(drop=True)

print("  TTLJ entries =", len(df_bkg))


In [ ]:
# ============================================================
# 7. Cleaning for inference
#
# +/-inf -> NaN
# <= -998 sentinel -> NaN
#
# Feature NaN 때문에 event를 drop하지 않습니다.
# 4번째 b jet이 없는 event도 그대로 inference에 들어갑니다.
# ============================================================
def clean_for_inference(df):
    out = df.copy()

    for feat in FEATURES:
        out[feat] = pd.to_numeric(out[feat], errors="coerce")
        out.loc[np.isinf(out[feat]), feat] = np.nan
        out.loc[out[feat] <= -998, feat] = np.nan

    out[WEIGHT_COL] = pd.to_numeric(out[WEIGHT_COL], errors="coerce")
    out = out[np.isfinite(out[WEIGHT_COL])].copy()

    return out

for ma in MA_VALUES:
    sig_dfs[ma] = clean_for_inference(sig_dfs[ma])

df_bkg = clean_for_inference(df_bkg)

print("After cleaning:")
for ma in MA_VALUES:
    print(f"  MA={ma:3d}: {len(sig_dfs[ma])}")
print("  TTLJ  :", len(df_bkg))


In [ ]:
# ============================================================
# 8. Sanity check
# ============================================================
def feature_sanity(df, name):
    rows = []
    for feat in FEATURES:
        x = df[feat]
        rows.append({
            "feature": feat,
            "nan_fraction": float(x.isna().mean()),
            "zero_fraction": float((x == 0).mean()),
        })

    check = pd.DataFrame(rows)
    print(f"\n===== {name} =====")
    display(
        check.sort_values(
            ["nan_fraction", "zero_fraction"],
            ascending=False
        ).head(20)
    )

feature_sanity(df_bkg, "TTLJ")

if 90 in sig_dfs:
    feature_sanity(sig_dfs[90], "Signal MA90")


In [ ]:
# ============================================================
# 9. Prepare ONNX input
# ============================================================
def transform_features(df):
    X = df[FEATURES].to_numpy(dtype=np.float32)

    if SCALER_TYPE is None:
        return X

    if scaler is not None:
        return scaler.transform(X).astype(np.float32)

    if SCALER_TYPE == "standard":
        mean = np.asarray(scaler_meta["mean"], dtype=np.float32)
        scale = np.asarray(scaler_meta["scale"], dtype=np.float32)
        return ((X - mean) / scale).astype(np.float32)

    if SCALER_TYPE == "minmax":
        min_ = np.asarray(scaler_meta["min"], dtype=np.float32)
        scale = np.asarray(scaler_meta["scale"], dtype=np.float32)
        return (X * scale + min_).astype(np.float32)

    raise ValueError(f"Unknown scaler type: {SCALER_TYPE}")


In [ ]:
# ============================================================
# 10. Load ONNX
# ============================================================
sess = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"],
)

print("ONNX inputs:")
for x in sess.get_inputs():
    print(" ", x.name, x.shape, x.type)

print("\nONNX outputs:")
for x in sess.get_outputs():
    print(" ", x.name, x.shape, x.type)

INPUT_NAME = sess.get_inputs()[0].name
print("\nUsing input:", INPUT_NAME)


In [ ]:
# ============================================================
# 11. Robust ONNX probability extraction
# ============================================================
def extract_signal_probability(outputs):
    # Binary XGBoost ONNX output can depend on package version.

    if len(outputs) == 1:
        arr = np.asarray(outputs[0])
        if arr.ndim == 2 and arr.shape[1] >= 2:
            return arr[:, 1].astype(float)
        return arr.reshape(-1).astype(float)

    prob = outputs[-1]

    if isinstance(prob, list):
        return np.asarray(
            [p[1] if 1 in p else p.get("1") for p in prob],
            dtype=float
        )

    arr = np.asarray(prob)

    if arr.ndim == 2 and arr.shape[1] >= 2:
        return arr[:, 1].astype(float)

    return arr.reshape(-1).astype(float)


def predict_onnx(df, batch_size=200000):
    X = transform_features(df)
    preds = []

    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        outputs = sess.run(None, {INPUT_NAME: X[start:stop]})
        preds.append(extract_signal_probability(outputs))

    return np.concatenate(preds)


In [ ]:
# ============================================================
# 12. SAME combined-MA ONNX -> every MA + TTLJ
# ============================================================
print("Running TTLJ inference ...")
df_bkg["bdt_score"] = predict_onnx(df_bkg)

for ma in MA_VALUES:
    print(f"Running MA={ma} inference ...")
    sig_dfs[ma]["bdt_score"] = predict_onnx(sig_dfs[ma])

print("\nScore ranges")
print(
    f"TTLJ: {df_bkg['bdt_score'].min():.5f} -> "
    f"{df_bkg['bdt_score'].max():.5f}"
)

for ma in MA_VALUES:
    s = sig_dfs[ma]["bdt_score"]
    print(f"MA={ma:3d}: {s.min():.5f} -> {s.max():.5f}")


In [ ]:
# ============================================================
# 13. Score distributions
# ============================================================
bins = np.linspace(0, 1, 51)

plt.figure(figsize=(9, 7))

plt.hist(
    df_bkg["bdt_score"],
    bins=bins,
    weights=np.abs(df_bkg[WEIGHT_COL]),
    density=True,
    histtype="step",
    linewidth=2,
    label="TTLJ",
)

for ma in MA_VALUES:
    df = sig_dfs[ma]
    plt.hist(
        df["bdt_score"],
        bins=bins,
        weights=np.abs(df[WEIGHT_COL]),
        density=True,
        histtype="step",
        linewidth=1.5,
        label=fr"$m_A={ma}$ GeV",
    )

plt.xlabel("Combined-MA BDT score")
plt.ylabel("Normalized events")
plt.title(f"MHc{TARGET_MHC}: combined-MA ONNX score")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()

plt.savefig(PLOT_DIR / "score_each_MA.png", dpi=180)
plt.savefig(PDF_DIR / "score_each_MA.pdf")
plt.show()


In [ ]:
# ============================================================
# 14. ROC/AUC for each MA
# ============================================================
roc_results = {}
auc_rows = []

bkg_score = df_bkg["bdt_score"].to_numpy()
bkg_weight = np.abs(df_bkg[WEIGHT_COL].to_numpy(dtype=float))

for ma in MA_VALUES:
    sig = sig_dfs[ma]

    sig_score = sig["bdt_score"].to_numpy()
    sig_weight = np.abs(sig[WEIGHT_COL].to_numpy(dtype=float))

    y_true = np.concatenate([
        np.ones(len(sig_score), dtype=int),
        np.zeros(len(bkg_score), dtype=int),
    ])

    y_score = np.concatenate([sig_score, bkg_score])
    weights = np.concatenate([sig_weight, bkg_weight])

    fpr, tpr, thresholds = roc_curve(
        y_true, y_score, sample_weight=weights
    )

    auc_value = roc_auc_score(
        y_true, y_score, sample_weight=weights
    )

    roc_results[ma] = {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "auc": auc_value,
    }

    auc_rows.append({
        "MA": ma,
        "AUC": auc_value,
        "N_signal": len(sig),
        "N_background": len(df_bkg),
    })

auc_df = pd.DataFrame(auc_rows)
display(auc_df)

print("Mean per-MA AUC =", auc_df["AUC"].mean())


In [ ]:
# ============================================================
# 15. Plot: For each mass
#
# 선배 그림의 오른쪽과 대응.
# 단, 지금은 MA별 model이 아니라 SAME combined model입니다.
# ============================================================
plt.figure(figsize=(8, 7))

for ma in MA_VALUES:
    r = roc_results[ma]
    plt.plot(
        r["fpr"],
        r["tpr"],
        linewidth=1.6,
        label=fr"$m_A={ma}$ GeV (AUC={r['auc']:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

plt.xlabel("Background Efficiency")
plt.ylabel("Signal Efficiency")
plt.title(f"Combined-MA model: each mass, MHc={TARGET_MHC} GeV")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(fontsize=9)
plt.tight_layout()

plt.savefig(PLOT_DIR / "roc_each_MA.png", dpi=180)
plt.savefig(PDF_DIR / "roc_each_MA.pdf")
plt.show()


In [ ]:
# ============================================================
# 16. Combine all MA signals with equal MA contribution
#
# event 수가 많은 MA가 overall ROC를 지배하지 않도록
# 각 MA의 total signal weight를 동일하게 normalize합니다.
# ============================================================
sig_all_parts = []

for ma in MA_VALUES:
    temp = sig_dfs[ma][["bdt_score", WEIGHT_COL]].copy()
    temp["mass_point"] = ma
    temp["eval_weight"] = np.abs(temp[WEIGHT_COL].astype(float))

    sumw = temp["eval_weight"].sum()
    if sumw <= 0:
        raise RuntimeError(f"Invalid signal weight sum for MA={ma}")

    temp["eval_weight_equal_ma"] = temp["eval_weight"] / sumw
    sig_all_parts.append(temp)

df_sig_all = pd.concat(sig_all_parts, ignore_index=True)

df_bkg_eval = df_bkg[["bdt_score", WEIGHT_COL]].copy()
df_bkg_eval["eval_weight"] = np.abs(
    df_bkg_eval[WEIGHT_COL].astype(float)
)

sig_equal_sumw = df_sig_all["eval_weight_equal_ma"].sum()
bkg_sumw = df_bkg_eval["eval_weight"].sum()

df_bkg_eval["eval_weight_equal_ma"] = (
    df_bkg_eval["eval_weight"] * sig_equal_sumw / bkg_sumw
)

y_true_all = np.concatenate([
    np.ones(len(df_sig_all), dtype=int),
    np.zeros(len(df_bkg_eval), dtype=int),
])

y_score_all = np.concatenate([
    df_sig_all["bdt_score"].to_numpy(),
    df_bkg_eval["bdt_score"].to_numpy(),
])

weight_all = np.concatenate([
    df_sig_all["eval_weight_equal_ma"].to_numpy(),
    df_bkg_eval["eval_weight_equal_ma"].to_numpy(),
])

fpr_all, tpr_all, _ = roc_curve(
    y_true_all, y_score_all, sample_weight=weight_all
)

auc_all = roc_auc_score(
    y_true_all, y_score_all, sample_weight=weight_all
)

print(f"Combined equal-MA signal AUC = {auc_all:.6f}")


In [ ]:
# ============================================================
# 17. Overall combined-MA ROC
# ============================================================
plt.figure(figsize=(7, 7))

plt.plot(
    fpr_all,
    tpr_all,
    linewidth=2,
    label=f"All MA combined (AUC={auc_all:.4f})",
)

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

plt.xlabel("Background Efficiency")
plt.ylabel("Signal Efficiency")
plt.title(f"Combined-MA ONNX: all masses, MHc={TARGET_MHC} GeV")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend()
plt.tight_layout()

plt.savefig(PLOT_DIR / "roc_all_MA_combined.png", dpi=180)
plt.savefig(PDF_DIR / "roc_all_MA_combined.pdf")
plt.show()


In [ ]:
# ============================================================
# 18. AUC vs MA
#
# combined model이 특정 MA에서만 잘 작동하는지 확인.
# ============================================================
plt.figure(figsize=(8, 5))

plt.plot(
    auc_df["MA"],
    auc_df["AUC"],
    marker="o",
)

plt.axhline(
    auc_df["AUC"].mean(),
    linestyle="--",
    label=f"mean = {auc_df['AUC'].mean():.4f}",
)

plt.xlabel(r"$m_A$ [GeV]")
plt.ylabel("ROC AUC")
plt.title(f"Combined-MA model performance vs mass (MHc={TARGET_MHC} GeV)")
plt.ylim(
    max(0.5, auc_df["AUC"].min() - 0.05),
    min(1.0, auc_df["AUC"].max() + 0.05),
)
plt.legend()
plt.tight_layout()

plt.savefig(PLOT_DIR / "auc_vs_MA.png", dpi=180)
plt.savefig(PDF_DIR / "auc_vs_MA.pdf")
plt.show()


In [ ]:
# ============================================================
# 19. Summary
# ============================================================
summary_df = auc_df.copy()
summary_df["delta_from_mean"] = (
    summary_df["AUC"] - summary_df["AUC"].mean()
)
summary_df = summary_df.sort_values("MA").reset_index(drop=True)

display(summary_df)

best_idx = summary_df["AUC"].idxmax()
worst_idx = summary_df["AUC"].idxmin()

print("\nBest:")
display(summary_df.loc[[best_idx]])

print("Worst:")
display(summary_df.loc[[worst_idx]])

print(f"Mean per-MA AUC         = {summary_df['AUC'].mean():.6f}")
print(f"Std of per-MA AUC       = {summary_df['AUC'].std(ddof=0):.6f}")
print(f"All-MA equal-weight AUC = {auc_all:.6f}")


In [ ]:
# ============================================================
# 20. Save numerical results
# ============================================================
auc_csv = OUTPUT_DIR / "auc_by_mass.csv"
summary_df.to_csv(auc_csv, index=False)

save_dict = {}
for ma in MA_VALUES:
    save_dict[f"MA{ma}_fpr"] = roc_results[ma]["fpr"]
    save_dict[f"MA{ma}_tpr"] = roc_results[ma]["tpr"]
    save_dict[f"MA{ma}_thresholds"] = roc_results[ma]["thresholds"]

save_dict["all_MA_fpr"] = fpr_all
save_dict["all_MA_tpr"] = tpr_all

roc_npz = OUTPUT_DIR / "roc_curves_by_mass.npz"
np.savez(roc_npz, **save_dict)

print("Saved:", auc_csv)
print("Saved:", roc_npz)


In [ ]:
# ============================================================
# 21. OPTIONAL: save event-level ONNX scores
# ============================================================
SAVE_EVENT_SCORES = False

if SAVE_EVENT_SCORES:
    bkg_out = df_bkg[["bdt_score", WEIGHT_COL]].copy()
    bkg_out["sample"] = "TTLJ"
    bkg_out["mass_point"] = np.nan

    pieces = [bkg_out]

    for ma in MA_VALUES:
        temp = sig_dfs[ma][["bdt_score", WEIGHT_COL]].copy()
        temp["sample"] = "AtoBB"
        temp["mass_point"] = ma
        pieces.append(temp)

    score_df = pd.concat(pieces, ignore_index=True)

    score_path = OUTPUT_DIR / "event_scores.csv"
    score_df.to_csv(score_path, index=False)

    print("Saved:", score_path)


## 결과를 볼 때

이번 combined-MA 모델에서는 특히 다음을 보면 됩니다.

- **`roc_each_MA`**: 같은 ONNX가 각 MA에서 separation을 유지하는가?
- **`auc_vs_MA`**: 특정 MA에서만 성능이 급격히 떨어지는가?
- **`roc_all_MA_combined`**: MA별 contribution을 동일하게 했을 때 전체 성능은 어떤가?
- **score shape**: mass에 따라 BDT score가 이상하게 이동하지 않는가?

### 선배의 예전 `Average`와 차이

예전 결과가 **MA별로 따로 학습한 여러 ONNX의 score를 평균**한 것이라면, 그것은 이번 combined-MA single model과 정의가 다릅니다.

이번 구조는:

`8개 MA signal → 하나의 training → 하나의 ONNX → 모든 MA에 같은 ONNX 적용`

입니다.

따라서 지금 우선 만들어야 하는 핵심 그림은 **For each mass + AUC vs MA**입니다.

이 notebook에서 성능을 확인한 다음, 검증된 ONNX를 SKNano/C++ analysis에 넣어 전체 MC/Data에 BDT score branch를 만드는 단계로 넘어가면 됩니다.
